# Introduction to Deep Learning with PyTorch — Part III: Pretrained Models & Finetuning

**Vanessa Gómez Verdejo,  Pablo Martínez Olmos, Alejandro Lancho Serrano, Emilio Parrado Hernández**  

Departamento de Teoría de la Señal y Comunicaciones  

**Universidad Carlos III de Madrid**

<img src="http://www.tsc.uc3m.es/~emipar/BBVA/INTRO/img/logo_uc3m_foot.jpg" width="400" />

## Learning Objectives — Part III

In this session we extend our workflow to **convolutional networks** and **transfer learning**.  
By the end of this notebook, you will be able to:

- **Implement and train a CNN** from scratch in PyTorch Lightning.  
- **Compare architectures** — MLP vs CNN — under the same MLflow tracking setup.  
- **Load and adapt a pretrained ResNet18** to a new task (MNIST).  
- Distinguish two transfer strategies:  
  - **Feature extraction** → freeze the backbone and train a new classifier head.  
  - **Fine-tuning** → unfreeze part or all of the network for joint training.  
- **Monitor and compare results** in MLflow: metrics, confusion matrices, and artifacts.

The goal is to see, in practice, how **transfer learning** can achieve high performance with minimal training data and computation compared to models trained from scratch.

---

## Recap from Part II

Previously, we:
- Introduced **PyTorch Lightning** as a clean framework for organizing training.  
- Integrated **MLflow** for experiment tracking and artifact logging.  
- Performed a short **hyperparameter sweep** comparing MLP runs.  
- Learned how to monitor training using both built-in and external loggers.


In [1]:
# --- Reused from Part I & II (no new material): data loaders + simple models ---
# This cell redefines the MNIST loaders
# so that Part III runs independently even if you start here.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(1)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- MNIST dataset & loaders ----------
def get_mnist_loaders(batch_size=64, val_ratio=0.1, root_dir="./data"):
    """Create train/val/test loaders for MNIST."""
    transform = transforms.ToTensor()

    full_train = datasets.MNIST(root=root_dir, train=True,  download=True, transform=transform)
    test_set   = datasets.MNIST(root=root_dir, train=False, download=True, transform=transform)

    num_train = len(full_train)
    val_size  = int(val_ratio * num_train)
    train_size = num_train - val_size

    train_set, val_set = random_split(full_train, [train_size, val_size])

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=pin)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_mnist_loaders()


print("Data & models ready. Example shapes:",
      next(iter(train_loader))[0].shape, "->", next(iter(train_loader))[1].shape)

Data & models ready. Example shapes: torch.Size([64, 1, 28, 28]) -> torch.Size([64])


#### Pytorch MLP class from Part II

In [2]:
# ---------- Baseline models ----------
class PyTorchMLP(nn.Module):
    """Simple MLP used in Part I (flatten -> Linear-ReLU-Linear)."""
    def __init__(self, num_features=28*28, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features, 50),
            nn.ReLU(),
            nn.Linear(50, 25),
            nn.ReLU(),
            nn.Linear(25, num_classes),
        )
    def forward(self, x):
        return self.net(x)

## Convolutional Neural Networks (CNNs)

Convolutional Neural Networks (CNNs) are deep models that exploit the **spatial structure** of images. Instead of treating every pixel independently (like an MLP), CNNs apply **learnable filters** that slide over the image to detect local patterns (edges, corners, strokes).

### Key ideas

- **Local receptive fields:** Each filter looks at a small patch (e.g., 3×3). Early layers learn primitive features; deeper layers combine them into more abstract shapes.
- **Weight sharing:** The same filter is reused across all spatial locations, drastically reducing parameters compared to fully connected layers.
- **Translation tolerance:** Convolutions scan the image, so the same feature can be detected anywhere.
- **Hierarchy of features:** Lower layers capture edges/curves; higher layers capture digit parts and full digits.

### Why CNNs for MNIST?

- Digits have strong **local structure** (strokes, junctions).
- CNNs are **more parameter-efficient** and often **generalize better** than MLPs on images.
- With similar training time, CNNs typically achieve **higher accuracy** on MNIST.

> In the next cells, we define a small CNN and train it on MNIST using plain PyTorch (the same loaders you created above).

In [3]:
import torch
import torch.nn as nn

class PyTorchCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Convolutional feature extractor
        self.conv_layers = nn.Sequential(
            # Input: [B, 1, 28, 28]
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),  # -> [B, 32, 28, 28]
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # -> [B, 32, 14, 14]

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # -> [B, 64, 14, 14]
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # -> [B, 64, 7, 7]
        )
        # Classifier head
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                     # -> [B, 64*7*7]
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),      # logits
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

**Padding, stride, pooling (quick intuition reminder)**  
- **Padding=1** with **kernel=3** keeps the spatial size (28→28), making shapes predictable.  
- **Stride=1** scans every pixel; larger strides downsample but may skip detail.  
- **MaxPool(2)** halves width/height (28→14→7) and keeps the strongest local activations, adding some translation tolerance while reducing compute.

## From MLP to CNN in Lightning

We already trained an MLP on MNIST in previous sessions.  
Now we’ll integrate our new **CNN** architecture into the same **Lightning + MLflow** workflow.

This allows us to compare both models directly:
- MLP → fast and simple, but limited in spatial awareness.  
- CNN → spatially structured, more accurate but slightly heavier.

We’ll reuse the same training pipeline:
1. Define a `LightningModule` that wraps our `PyTorchCNN`.  
2. Train for a few epochs on MNIST.  
3. Log results automatically to MLflow for later comparison.

---

In [4]:
# --- LightningModule for the CNN (reusing structure from Part II) ---

import pytorch_lightning as pl
from torchmetrics.classification import Accuracy

class LightningCNN(pl.LightningModule):
    def __init__(self, model, learning_rate=1e-3, num_classes=10):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model

        # Accuracy metrics per split (same style as before)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc   = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc  = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        features, labels = batch
        logits = self(features)
        loss = F.cross_entropy(logits, labels)
        preds = torch.argmax(logits, dim=1)
        return loss, preds, labels

    def training_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.log("train_loss", loss)
        self.train_acc(preds, labels)
        self.log("train_acc", self.train_acc, on_epoch=True, on_step=False, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.log("val_loss", loss, prog_bar=True)
        self.val_acc(preds, labels)
        self.log("val_acc", self.val_acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.test_acc(preds, labels)
        self.log("accuracy", self.test_acc)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

### Training the CNN

We’ll now train the CNN using the same pipeline as before:
- a `Trainer` from PyTorch Lightning,  
- integrated MLflow logger,  
- automatic metric and artifact tracking.

This will generate a **new MLflow run** under the same “MNIST” experiment,
so we can directly compare it with our previous MLP results.

In [5]:
from pytorch_lightning.loggers import MLFlowLogger
import mlflow

# --- MLflow setup ---
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("MNIST")

# --- Define hyperparameters once ---
learning_rate = 1e-3
num_epochs = 5
num_classes = 10

# --- Build a descriptive run name dynamically ---
run_name = f"CNN-lr={learning_rate}-ep={num_epochs}"

# --- MLflow logger ---
mlf_logger_cnn = MLFlowLogger(
    experiment_name="MNIST",
    tracking_uri="file:./mlruns",
    run_name=run_name
)

# --- Instantiate model and LightningModule ---
cnn_model = PyTorchCNN(num_classes=num_classes)
lit_cnn = LightningCNN(model=cnn_model, learning_rate=learning_rate, num_classes=num_classes)

# --- Trainer configuration ---
trainer_cnn = pl.Trainer(
    max_epochs=num_epochs,
    accelerator="auto",
    devices="auto",
    deterministic=True,
    log_every_n_steps=50,
    logger=mlf_logger_cnn
)

# --- Train & validate CNN ---
trainer_cnn.fit(lit_cnn, train_dataloaders=train_loader, val_dataloaders=val_loader)

# --- Evaluate on test set ---
test_metrics_cnn = trainer_cnn.test(lit_cnn, dataloaders=test_loader)
print("Test metrics:", test_metrics_cnn)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | PyTorchCNN         | 421 K  | train
1 | train_acc | MulticlassAccuracy | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
421 K     Trainable params
0         Non-trainable params
421 K     Total params
1.687     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:428: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Epoch 4: 100%|██████████| 844/844 [00:14<00:00, 59.70it/s, v_num=0306, val_loss=0.0346, val_acc=0.991, train_acc=0.994]

...

Epoch 4: 100%|██████████| 844/844 [00:14<00:00, 59.60it/s, v_num=0306, val_loss=0.0346, val_acc=0.991, train_acc=0.994]


/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:428: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


Testing DataLoader 0: 100%|██████████| 157/157 [00:01<00:00, 121.92it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         accuracy          │    0.9908999800682068     │
└───────────────────────────┴───────────────────────────┘

Test metrics: [{'accuracy': 0.9908999800682068}]


## From CNNs to Pretrained Models and Transfer Learning

Our small CNN already performs well on MNIST, but it was trained **from scratch** —  
every weight was initialized randomly and learned only from digit data.

For more complex datasets (like ImageNet or CIFAR), training from scratch is often:
- **Slow** — requires millions of images and hours of GPU time.  
- **Data-hungry** — needs huge datasets to avoid overfitting.  
- **Redundant** — low-level features (edges, textures) are similar across tasks.

This motivates **transfer learning**:  
we start from a model pretrained on a large dataset (e.g., ImageNet) and reuse its learned representations.

---

### Two main strategies

| Strategy | What happens | When to use |
|-----------|---------------|-------------|
| **Feature extraction** | Freeze the convolutional “backbone” (no gradient updates) and train only a new classification head. | When your dataset is small or very different from ImageNet (digits, medical images, etc.). |
| **Fine-tuning** | Unfreeze some or all layers and keep training with a smaller learning rate. | When your dataset is large or somewhat similar to ImageNet. |

Both methods allow us to leverage **general visual knowledge** captured by large models like ResNet, while adapting them to our specific task.

In the next cells, we’ll load a **pretrained ResNet-18**, adapt it for MNIST (1-channel grayscale images, 10 classes), and compare it against our MLP and CNN baselines — all under the same MLflow experiment.

### Feature Extraction: Quick design notes

1. **Respect the pretrained input distribution**  
   ResNet-18 was trained on **3-channel RGB, 224×224** images with **ImageNet normalization**.  
   We **keep the original `conv1` (3 channels)** and instead adapt MNIST with transforms:  
   `Grayscale(num_output_channels=3) → Resize(224,224) → ToTensor → Normalize(ImageNet mean/std)`.

2. **Output adaptation**  
   The ImageNet head outputs 1000 classes.  
   We **replace `fc` with `Linear(num_features, 10)`** to classify MNIST digits.

3. **Which weights are trained?**  
   - **Feature extraction:** freeze the backbone (all layers except `fc`). Only the new classifier head learns.  
   - **Fine-tuning:** unfreeze part/all of the backbone to adapt features to MNIST.

4. **Initialization**  
   All **pretrained layers** keep their ImageNet weights.  
   Only the **new `fc` layer** starts randomly initialized (standard PyTorch init).

---

Next, we’ll wrap this adapted model into a `LightningModule`, first in **feature extraction** mode (frozen backbone), then in **fine-tuning** mode.

In [11]:
# --- Load and adapt a pretrained ResNet18 for MNIST ---

from torchvision import models, transforms
import torch.nn as nn

# 1. Load pretrained ResNet18 on ImageNet
# ----------------------------------------------------------
# These weights were trained on 3-channel (RGB) 224×224 images.
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. KEEP the original 3-channel first convolution.
# ----------------------------------------------------------
# Instead of modifying conv1 (which would destroy pretrained filters),
# we will adapt MNIST images to 3 channels using torchvision transforms.
# So conv1 stays as-is: in_channels=3, out_channels=64, kernel=7×7, stride=2.

# 3. Replace the final classification head for MNIST (10 classes).
# ----------------------------------------------------------
# The pretrained classifier outputs 1000 ImageNet categories.
# We substitute it with a new Linear layer that outputs 10 digits.
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 10)

print("ResNet18 ready — first conv kept (3-channel input), new fc for 10 classes.")
print(resnet18.fc)

ResNet18 ready — first conv kept (3-channel input), new fc for 10 classes.
Linear(in_features=512, out_features=10, bias=True)


In [12]:
# --- Transform pipeline for MNIST adapted to pretrained ResNet18 ---
# This ensures inputs "look like" ImageNet images (3 channels, 224x224, normalized).

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)),         # ResNet expects 224×224 input
    transforms.Grayscale(num_output_channels=3),  # repeat channel -> RGB
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Rebuild loaders with new transforms
train_loader, val_loader, test_loader = get_mnist_loaders()
train_loader.dataset.dataset.transform = transform_resnet
val_loader.dataset.dataset.transform   = transform_resnet
test_loader.dataset.transform          = transform_resnet

print("MNIST transforms adapted for pretrained ResNet (3×224×224 + normalization).")

MNIST transforms adapted for pretrained ResNet (3×224×224 + normalization).


In [13]:
# --- LightningModule for transfer learning (Feature Extraction mode) ---

class LitResNet(pl.LightningModule):
    def __init__(self, model, lr=1e-3, num_classes=10, freeze_backbone=True):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.freeze_backbone = freeze_backbone

        # Accuracy metrics
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc   = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc  = Accuracy(task="multiclass", num_classes=num_classes)

        # 🔒 Freeze all pretrained layers except the final classifier (fc)
        if self.freeze_backbone:
            for name, param in self.model.named_parameters():
                if not name.startswith("fc"):
                   param.requires_grad = False
        else:
            # ensure everything is trainable
            for p in self.model.parameters():
                p.requires_grad = True

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        features, labels = batch
        logits = self(features)
        loss = F.cross_entropy(logits, labels)
        preds = torch.argmax(logits, dim=1)
        return loss, preds, labels

    def training_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.log("train_loss", loss)
        self.train_acc(preds, labels)
        self.log("train_acc", self.train_acc, on_epoch=True, on_step=False, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.log("val_loss", loss, prog_bar=True)
        self.val_acc(preds, labels)
        self.log("val_acc", self.val_acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        loss, preds, labels = self._shared_step(batch)
        self.test_acc(preds, labels)
        self.log("accuracy", self.test_acc)

    def configure_optimizers(self):
        # Only train parameters that are unfrozen
        return torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.hparams.lr
        )

### Training the pretrained model (feature extraction)

We’ll begin with **feature extraction mode** — only the new classification head is trained.  
This runs very fast and should already outperform our small CNN on MNIST.

We’ll log everything in the same **MLflow “MNIST” experiment** so that we can compare all three models later (MLP, CNN, and ResNet-18 feature extractor).

In [14]:
from pytorch_lightning.loggers import MLFlowLogger
import mlflow

# --- MLflow setup ---
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("MNIST")

# --- Define hyperparameters once ---
learning_rate = 1e-3
num_epochs = 5
freeze_backbone = True

# --- Dynamic run name for clarity in MLflow ---
run_name = f"ResNet18-lr={learning_rate}-ep={num_epochs}-frozen={freeze_backbone}"

# --- MLflow logger with descriptive run name ---
mlf_logger_resnet = MLFlowLogger(
    experiment_name="MNIST",
    tracking_uri="file:./mlruns",
    run_name=run_name
)

# --- Instantiate LightningModule (frozen backbone) ---
lit_resnet_frozen = LitResNet(
    model=resnet18,
    lr=learning_rate,
    freeze_backbone=freeze_backbone
)

# --- Trainer configuration ---
trainer_resnet_frozen = pl.Trainer(
    max_epochs=num_epochs,
    accelerator="auto",
    devices="auto",
    deterministic=True,
    log_every_n_steps=50,
    logger=mlf_logger_resnet
)

# --- Train (only classifier head will update) ---
trainer_resnet_frozen.fit(lit_resnet_frozen, train_dataloaders=train_loader, val_dataloaders=val_loader)

# --- Evaluate on test set ---
trainer_resnet_frozen.test(lit_resnet_frozen, dataloaders=test_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | ResNet             | 11.2 M | train
1 | train_acc | MulticlassAccuracy | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
5.1 K     Trainable params
11.2 M    Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
71        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 844/844 [01:36<00:00,  8.77it/s, v_num=6b57, val_loss=0.123, val_acc=0.962, train_acc=0.961]

`Trainer.fit` stopped: `max_epochs=5` reached.


Testing DataLoader 0: 100%|██████████| 157/157 [00:13<00:00, 11.23it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         accuracy          │    0.9634000062942505     │
└───────────────────────────┴───────────────────────────┘

[{'accuracy': 0.9634000062942505}]

### Fine-tuning the pretrained model

Next, we unfreeze the backbone so the entire ResNet can adapt to MNIST.  
We’ll train with a smaller learning rate (e.g., `1e-4`) to avoid disrupting the pretrained weights too quickly.

In [15]:
from pytorch_lightning.loggers import MLFlowLogger
import mlflow

# --- MLflow setup ---
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("MNIST")

# --- Define hyperparameters once ---
learning_rate = 1e-4
num_epochs = 5
freeze_backbone = False

# --- Dynamic, descriptive run name ---
run_name = f"ResNet18-lr={learning_rate}-ep={num_epochs}-frozen={freeze_backbone}"

# --- MLflow logger with run name ---
mlf_logger_resnet_ft = MLFlowLogger(
    experiment_name="MNIST",
    tracking_uri="file:./mlruns",
    run_name=run_name
)

# --- Instantiate LightningModule (fine-tuning: backbone unfrozen) ---
lit_resnet_finetune = LitResNet(
    model=resnet18,
    lr=learning_rate,
    freeze_backbone=freeze_backbone
)

# --- Trainer configuration ---
trainer_resnet_ft = pl.Trainer(
    max_epochs=num_epochs,
    accelerator="auto",
    devices="auto",
    deterministic=True,
    log_every_n_steps=50,
    logger=mlf_logger_resnet_ft
)

# --- Train (fine-tune the entire model) ---
trainer_resnet_ft.fit(lit_resnet_finetune, train_dataloaders=train_loader, val_dataloaders=val_loader)

# --- Evaluate on test set ---
trainer_resnet_ft.test(lit_resnet_finetune, dataloaders=test_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | ResNet             | 11.2 M | train
1 | train_acc | MulticlassAccuracy | 0      | train
2 | val_acc   | MulticlassAccuracy | 0      | train
3 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.727    Total estimated model params size (MB)
71        Modules in train mode
0         Modules in eval mode


Epoch 4: 100%|██████████| 844/844 [04:49<00:00,  2.92it/s, v_num=d347, val_loss=0.0254, val_acc=0.994, train_acc=0.997]

`Trainer.fit` stopped: `max_epochs=5` reached.


Testing DataLoader 0: 100%|██████████| 157/157 [00:14<00:00, 11.12it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         accuracy          │    0.9922000169754028     │
└───────────────────────────┴───────────────────────────┘

[{'accuracy': 0.9922000169754028}]

## Summary and Takeaways

In this session we extended our MLflow + Lightning workflow to **convolutional and pretrained networks**.

**What we achieved:**
- Implemented and trained a **CNN** from scratch for MNIST.  
- Integrated the CNN and MLP baselines into the **same MLflow experiment** for easy comparison.  
- Loaded a **pretrained ResNet-18**, adapted it to grayscale digits, and tested two modes:  
  - **Feature extraction:** freeze the backbone and train only the classifier.  
  - **Fine-tuning:** unfreeze layers to further adapt representations.  
- Observed that even a frozen ResNet backbone achieves **≈96 % accuracy** after a few epochs.  

**Key lessons:**
- CNNs leverage spatial structure and are far more efficient for image tasks than MLPs.  
- Pretrained models offer strong transferable features, drastically reducing training effort.  
- Combining **Lightning** for clean training loops and **MLflow** for tracking creates a solid,
  scalable workflow for experimentation.

---

**Next steps:**  
You can extend this notebook by fine-tuning more layers, experimenting with data augmentation,
or applying the same workflow to a different dataset.